In [4]:
import pandas as pd

df = pd.read_csv('../../archive/yellow_tripdata_2015-01.csv')

print(f"Broj redova: {df.shape[0]}")
print(f"Broj kolona: {df.shape[1]}")

Broj redova: 12748986
Broj kolona: 19


In [5]:
print("PROVERA NEDOSTAJUCIH VREDNOSTI")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])


PROVERA NEDOSTAJUCIH VREDNOSTI
improvement_surcharge    3
dtype: int64


In [6]:
print("PROVERA EKSTREMNIH VREDNOSTI (OUTLIER-A)")

zero_distance = df[df['trip_distance'] == 0]
print(f"Vožnje sa 0 milja: {len(zero_distance):,}")

long_distance = df[df['trip_distance'] > 100]
print(f"Vožnje preko 100 milja: {len(long_distance):,}")

zero_fare = df[df['fare_amount'] <= 0]
print(f"Vožnje sa iznosom <= 0: {len(zero_fare):,}")

high_fare = df[df['fare_amount'] > 500]
print(f"Vožnje sa iznosom > 500$: {len(high_fare):,}")

invalid_passengers = df[(df['passenger_count'] == 0) | (df['passenger_count'] > 6)]
print(f"Vožnje sa neispravnim brojem putnika (0 ili >6): {len(invalid_passengers):,}")

negative_tip = df[df['tip_amount'] < 0]
print(f"Vožnje sa negativnom napojnicom: {len(negative_tip):,}")

df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
invalid_time = df[df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']]
print(f"Vožnje sa dropoff pre pickup: {len(invalid_time):,}")


PROVERA EKSTREMNIH VREDNOSTI (OUTLIER-A)
Vožnje sa 0 milja: 79,365
Vožnje preko 100 milja: 172
Vožnje sa iznosom <= 0: 7,666
Vožnje sa iznosom > 500$: 77
Vožnje sa neispravnim brojem putnika (0 ili >6): 6,595
Vožnje sa negativnom napojnicom: 79
Vožnje sa dropoff pre pickup: 297


In [7]:
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

print("PROVERA NEKONZISTENTNIH ZAPISA")

invalid_time = df[df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']]
print(f"Vožnje sa dropoff pre pickup: {len(invalid_time)}")

df['duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
long_rides = df[df['duration_minutes'] > 1440]  # 24h = 1440 min
print(f"Vožnje duže od 24h: {len(long_rides)}")

zero_duration = df[df['duration_minutes'] == 0]
print(f"Vožnje koje traju 0 minuta: {len(zero_duration)}")


PROVERA NEKONZISTENTNIH ZAPISA
Vožnje sa dropoff pre pickup: 297
Vožnje duže od 24h: 79
Vožnje koje traju 0 minuta: 14816


In [5]:
import pandas as pd
from sqlalchemy import create_engine
import pyodbc

server = 'DESKTOP-64J7LMV\\MSSQLSERVER2022' 
database = 'SkladistenjePROJEKAT'           

engine = create_engine(
    f'mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&Trusted_Connection=yes'
)

try:
    with engine.connect() as conn:
        print("Povezano!")
except Exception as e:
    print(f"Greška! {e}")



Povezano!


In [9]:
# Konvertujem vreme
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

# Računanje trajanja vožnje
df['trip_duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

# Čišćenje
df_clean = df[
    (df['trip_distance'] > 0) & 
    (df['trip_distance'] <= 100) & 
    (df['fare_amount'] > 0) & 
    (df['fare_amount'] <= 500) & 
    (df['passenger_count'] >= 1) & 
    (df['passenger_count'] <= 6) & 
    (df['tip_amount'] >= 0) &
    (df['tpep_dropoff_datetime'] >= df['tpep_pickup_datetime']) &
    (df['trip_duration_minutes'] > 0) &
    (df['trip_duration_minutes'] <= 1440) &
     (df['improvement_surcharge'].notna()) 
]

print(f"Broj zapisa pre čišćenja: {len(df):,}")
print(f"Broj zapisa posle čišćenja: {len(df_clean):,}")


Broj zapisa pre čišćenja: 12,748,986
Broj zapisa posle čišćenja: 12,657,278


In [12]:
putanja = r'C:\Users\Andjela\Desktop\yellow_tripdata_2015-01_OCISCEN.csv'
df_clean.to_csv(putanja, index=False)
print(f" Fajl sačuvan na: {putanja}")

 Fajl sačuvan na: C:\Users\Andjela\Desktop\yellow_tripdata_2015-01_OCISCEN.csv


In [ ]:
# DIM_DATE
dim_date = df_clean[['tpep_pickup_datetime']].copy()
dim_date['Date_ID'] = dim_date['tpep_pickup_datetime'].dt.strftime('%Y%m%d').astype(int)
dim_date['Full_Date'] = dim_date['tpep_pickup_datetime'].dt.date
dim_date['Year'] = dim_date['tpep_pickup_datetime'].dt.year
dim_date['Month'] = dim_date['tpep_pickup_datetime'].dt.month
dim_date['Month_Name'] = dim_date['tpep_pickup_datetime'].dt.strftime('%B')  # Naziv meseca
dim_date['Day'] = dim_date['tpep_pickup_datetime'].dt.day
dim_date['Day_Of_Week'] = dim_date['tpep_pickup_datetime'].dt.dayofweek + 1  # 1=Monday
dim_date['Day_Name'] = dim_date['tpep_pickup_datetime'].dt.strftime('%A')  # Naziv dana
dim_date['Is_Weekend'] = (dim_date['Day_Of_Week'] >= 6).astype(int)  # 1=vikend, 0=radni dan

dim_date.drop_duplicates(subset=['Date_ID'], inplace=True)

# Izbacujem tpep_pickup_datetime
dim_date_to_sql = dim_date[['Date_ID', 'Full_Date', 'Year', 'Month', 'Month_Name', 
                            'Day', 'Day_Of_Week', 'Day_Name', 'Is_Weekend']]

dim_date_to_sql.to_sql('DIM_DATE', engine, if_exists='append', index=False)
print(f"DIM_DATE popunjena sa {len(dim_date_to_sql):,} redova")

In [10]:
#DIM_TIME
dim_time = df_clean[['tpep_pickup_datetime']].copy()
dim_time['Time_ID'] = dim_time['tpep_pickup_datetime'].dt.hour * 100 + dim_time['tpep_pickup_datetime'].dt.minute
dim_time['Hour'] = dim_time['tpep_pickup_datetime'].dt.hour
dim_time['Minute'] = dim_time['tpep_pickup_datetime'].dt.minute

def get_time_of_day(hour):
    if 5 <= hour < 12:
        return 'Jutro'
    elif 12 <= hour < 17:
        return 'Podne'
    elif 17 <= hour < 21:
        return 'Veče'
    else:
        return 'Noć'

dim_time['Time_Of_Day'] = dim_time['Hour'].apply(get_time_of_day)
dim_time.drop_duplicates(subset=['Time_ID'], inplace=True)

# Biraj samo kolone koje postoje u tabeli
dim_time_to_sql = dim_time[['Time_ID', 'Hour', 'Minute', 'Time_Of_Day']]

dim_time_to_sql.to_sql('DIM_TIME', engine, if_exists='append', index=False)
print(f"DIM_TIME popunjena sa {len(dim_time_to_sql):,} redova")

DIM_TIME popunjena sa 1,440 redova


In [11]:
#DIM_PICKUP_LOCATION
dim_pickup = df_clean[['pickup_longitude', 'pickup_latitude']].rename(
    columns={'pickup_longitude': 'Longitude', 'pickup_latitude': 'Latitude'}
).drop_duplicates().dropna()

dim_pickup.to_sql('DIM_PICKUP_LOCATION', engine, if_exists='append', index=False)
print(f"DIM_PICKUP_LOCATION popunjena sa {len(dim_pickup):,} redova")

DIM_PICKUP_LOCATION popunjena sa 9,586,549 redova


In [12]:
#DIM_DROPOFF_LOCATION
dim_dropoff = df_clean[['dropoff_longitude', 'dropoff_latitude']].rename(
    columns={'dropoff_longitude': 'Longitude', 'dropoff_latitude': 'Latitude'}
).drop_duplicates().dropna()

dim_dropoff.to_sql('DIM_DROPOFF_LOCATION', engine, if_exists='append', index=False)
print(f"DIM_DROPOFF_LOCATION popunjena sa {len(dim_dropoff):,} redova")

DIM_DROPOFF_LOCATION popunjena sa 10,500,560 redova


In [14]:
#DIM_VENDOR
vendor_data = pd.DataFrame([
    [1, 'Creative Mobile Technologies'],
    [2, 'VeriFone Inc.']
], columns=['Vendor_Code', 'Vendor_Name'])

# Proveri da li tabela već ima podatke
existing_vendor = pd.read_sql("SELECT COUNT(*) AS broj FROM DIM_VENDOR", engine)
if existing_vendor['broj'][0] == 0:
    vendor_data.to_sql('DIM_VENDOR', engine, if_exists='append', index=False)
    print("DIM_VENDOR popunjena")
else:
    print("DIM_VENDOR već ima podatke")

#DIM_RATE_CODE
rate_data = pd.DataFrame([
    [1, 'Standard rate'],
    [2, 'JFK'],
    [3, 'Newark'],
    [4, 'Nassau or Westchester'],
    [5, 'Negotiated fare'],
    [6, 'Group ride']
], columns=['Rate_Code', 'Rate_Code_Name'])

existing_rate = pd.read_sql("SELECT COUNT(*) AS broj FROM DIM_RATE_CODE", engine)
if existing_rate['broj'][0] == 0:
    rate_data.to_sql('DIM_RATE_CODE', engine, if_exists='append', index=False)
    print("DIM_RATE_CODE popunjena")
else:
    print("DIM_RATE_CODE već ima podatke")

# DIM_PAYMENT_TYPE
payment_data = pd.DataFrame([
    [1, 'Credit card'],
    [2, 'Cash'],
    [3, 'No charge'],
    [4, 'Dispute'],
    [5, 'Unknown'],
    [6, 'Voided trip']
], columns=['Payment_Code', 'Payment_Name'])

existing_payment = pd.read_sql("SELECT COUNT(*) AS broj FROM DIM_PAYMENT_TYPE", engine)
if existing_payment['broj'][0] == 0:
    payment_data.to_sql('DIM_PAYMENT_TYPE', engine, if_exists='append', index=False)
    print("DIM_PAYMENT_TYPE popunjena")
else:
    print("DIM_PAYMENT_TYPE već ima podatke")

DIM_VENDOR već ima podatke
DIM_RATE_CODE već ima podatke
DIM_PAYMENT_TYPE već ima podatke


In [15]:
#UČITAVANJE ID-jeva IZ DIMENZIJA
date_map = pd.read_sql("SELECT Date_ID, Full_Date FROM DIM_DATE", engine)
time_map = pd.read_sql("SELECT Time_ID, Hour, Minute FROM DIM_TIME", engine)
pickup_map = pd.read_sql("SELECT Pickup_Location_ID, Longitude, Latitude FROM DIM_PICKUP_LOCATION", engine)
dropoff_map = pd.read_sql("SELECT Dropoff_Location_ID, Longitude, Latitude FROM DIM_DROPOFF_LOCATION", engine)
vendor_map = pd.read_sql("SELECT Vendor_ID, Vendor_Code FROM DIM_VENDOR", engine)
rate_map = pd.read_sql("SELECT RateCode_ID, Rate_Code FROM DIM_RATE_CODE", engine)
payment_map = pd.read_sql("SELECT Payment_ID, Payment_Code FROM DIM_PAYMENT_TYPE", engine)

print("ID-jevi učitani")

ID-jevi učitani


In [16]:
#FAKTORSKA TABELA
fact = df_clean.copy()

# Dodaj ID-jeve za datum i vreme
fact['Date_ID'] = fact['tpep_pickup_datetime'].dt.strftime('%Y%m%d').astype(int)
fact['Time_ID'] = fact['tpep_pickup_datetime'].dt.hour * 100 + fact['tpep_pickup_datetime'].dt.minute

# Mapiranje lokacija (spajanje po koordinatama)
fact = fact.merge(pickup_map, left_on=['pickup_longitude', 'pickup_latitude'], 
                  right_on=['Longitude', 'Latitude'], how='left')
fact = fact.merge(dropoff_map, left_on=['dropoff_longitude', 'dropoff_latitude'], 
                  right_on=['Longitude', 'Latitude'], how='left')

# Mapiranje Vendor, Rate, Payment
fact = fact.merge(vendor_map, left_on='VendorID', right_on='Vendor_Code', how='left')
fact = fact.merge(rate_map, left_on='RateCodeID', right_on='Rate_Code', how='left')
fact = fact.merge(payment_map, left_on='payment_type', right_on='Payment_Code', how='left')

# Odaberi kolone koje FACT_TRIP tabela očekuje
fact_final = fact[[
    'Date_ID', 'Time_ID', 
    'Pickup_Location_ID', 'Dropoff_Location_ID',
    'Vendor_ID', 'RateCode_ID', 'Payment_ID',
    'passenger_count', 'trip_distance', 'fare_amount', 
    'extra', 'mta_tax', 'tip_amount', 
    'improvement_surcharge', 'total_amount', 
    'trip_duration_minutes'
]]

# Provera da nema NULL vrednosti u ključnim kolonama
print(f"Broj redova pre provere NULL: {len(fact_final):,}")
fact_final = fact_final.dropna(subset=['Date_ID', 'Time_ID', 'Pickup_Location_ID', 
                                       'Dropoff_Location_ID', 'Vendor_ID', 
                                       'RateCode_ID', 'Payment_ID'])
print(f"Broj redova posle uklanjanja NULL: {len(fact_final):,}")

# Učitaj u SQL Server
fact_final.to_sql('FACT_TRIP', engine, if_exists='append', index=False)
print(f" FACT_TRIP popunjena sa {len(fact_final):,} redova")

Broj redova pre provere NULL: 12,657,278
Broj redova posle uklanjanja NULL: 200,629
 FACT_TRIP popunjena sa 200,629 redova


In [2]:
import os

print("Trenutni radni direktorijum:", os.getcwd())

df_clean.to_csv('yellow_tripdata_2015-01_OCISCEN.csv', index=False)


if os.path.exists('yellow_tripdata_2015-01_clean.csv'):
    print(" Fajl je uspešno sačuvan!")
else:
    print("Fajl nije sačuvan. Proveri dozvole za pisanje.")

Trenutni radni direktorijum: c:\Users\Andjela\Documents\SkladistenjeProjekat\repo\SkladistenjeProjekat


NameError: name 'df_clean' is not defined